# RCC miRNA inference example

**CatBoost + TabM + ResNet + Ridge**.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# FINAL_VERSION - dir with config files
FINAL_VERSION = Path.cwd().resolve().parent
if str(FINAL_VERSION) not in sys.path:
    sys.path.insert(0, str(FINAL_VERSION))

from preprocessor import SingleCell
from TOTAL_INFERENCE.constants import MANIFEST_PATH

RCC_DIR = Path.cwd() / "RCC"
OUTPUT_DIR = Path.cwd() / "results_rcc"
OUTPUT_DIR.mkdir(exist_ok=True)

print("FINAL_VERSION:", FINAL_VERSION)
print("RCC files:", sorted(p.name for p in RCC_DIR.glob("RCC_S*.csv")))
print("Manifest:", MANIFEST_PATH)

FINAL_VERSION: /workspace/FINAL_VERSION
RCC files: ['RCC_S1.csv', 'RCC_S2.csv', 'RCC_S3.csv', 'RCC_S4.csv', 'RCC_S5.csv']
Manifest: /workspace/FINAL_VERSION/FINAL_CONFIG_AND_FIGURES/predict_manifest.json


## 1. Manifest и когорты miRNA

- **K1** — single-cell (after KNN impute), 79 miRNA
- **K2…K10** — pseudobulk KNN (without imputing), 89 miRNA
- Finally **168 eligible** miRNA (R² ≥ 0.4)

In [3]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

cohort_sizes = {k: len(v) for k, v in manifest["cohorts"].items()}
print("Eligible miRNAs:", len(manifest["eligible_mirs"]))
print("Cohort sizes:", cohort_sizes)

info = manifest["targets"]["hsa-mir-21-5p"]
print("\nhsa-mir-21-5p:")
print("  assigned_cohort:", info["assigned_cohort"])
print("  test R2 K1:", info["test_r2"]["K1"])
print("  n_genes:", len(info["genes"]))

Eligible miRNAs: 168
Cohort sizes: {'K1': 79, 'K2': 34, 'K3': 30, 'K4': 10, 'K5': 5, 'K10': 10}

hsa-mir-21-5p:
  assigned_cohort: K1
  test R2 K1: 0.94635226746475
  n_genes: 278


## 2. Load RCC dataset

Input: **cells × genes** CSV with columns `barcode`, `CellType` except for mRNA expression data (raw - counts)

In [8]:
SAMPLE = "RCC_S1"  # RCC_S1 … RCC_S5
INPUT_PATH = RCC_DIR / f"{SAMPLE}.csv"

# Для быстрой отладки: nrows=200. Для полного инференса: nrows=None
NROWS = None

raw = pd.read_csv(INPUT_PATH, nrows=NROWS)
raw = raw.set_index("barcode")

print(raw.shape)
print(raw[["CellType"]].head())
print("ENSG columns:", sum(str(c).startswith("ENSG") for c in raw.columns))

(6761, 33435)
                           CellType
barcode                            
AAACCCAGTAAGCAAT-1          T cells
AAACCCAGTCTGTGCG-1  Malignant cells
AAACCCAGTTAGAGAT-1          T cells
AAACCCATCACCTTGC-1          T cells
AAACGAAAGCGATTCT-1          T cells
ENSG columns: 33434


In [10]:
raw.head()

,CellType,ENSG00000243485,ENSG00000237613,ENSG00000186092,ENSG00000238009,ENSG00000239945,ENSG00000239906,ENSG00000241599,ENSG00000236601,ENSG00000284733,...,ENSG00000277196,ENSG00000277630,ENSG00000278384,ENSG00000278633,ENSG00000276345,ENSG00000277856,ENSG00000275063,ENSG00000271254,ENSG00000277475,ENSG00000268674
barcode,,,,,,,,,,,,,,,,,,,,,
AAACCCAGTAAGCAAT-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCCAGTCTGTGCG-1,Malignant cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCCAGTTAGAGAT-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
AAACCCATCACCTTGC-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACGAAAGCGATTCT-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. `SingleCell` and `StackPredictor` classes

Class includes standard preprocessing of data: (TPM, KNN-imputing, KNN pseudobulk sampling) и `StackPredictor` - prediction model

In [4]:
sc = SingleCell(
    device="cuda",        
    catboost_task="CPU",  
    preload_models=False,
)

print("K1 miRNAs:", len(sc.mirnas_for_cohort("K1")))
print("K2 miRNAs:", len(sc.mirnas_for_pseudobulk_k(2)))

K1 miRNAs: 79
K2 miRNAs: 34


## 4. Preprocessing step by step (single-cell K1)

### Шаг 4.1 — align genes

`prepare_input()` adjusts matrix to fix set of mRNA **17 392 ENSG**, non existing genes filled by 0.

In [15]:
counts_gc = sc.prepare_input(raw)
print("genes × cells:", counts_gc.shape)
print("gene order fixed:", counts_gc.index[:3].tolist())

genes × cells: (17392, 6761)
gene order fixed: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419']


### Шаг 4.2 — TPM and log2 normalization

Normalization: counts → RPK → TPM → `log2(TPM + 1)`.

In [16]:
log_tpm_gc = sc.TPM(counts_gc, enforce_mrna_standard=False)
log_tpm_cells = log_tpm_gc.T  # cells × genes — формат для stack

print("log2(TPM+1) cells × genes:", log_tpm_cells.shape)
log_tpm_cells.iloc[:3, :3]

✔ Found length for 17392/17392 genes (100.00%)
log2(TPM+1) cells × genes: (6761, 17392)


,ENSG00000000003,ENSG00000000005,ENSG00000000419
barcode,,,
AAACCCAGTAAGCAAT-1,0.0,0.0,0.0
AAACCCAGTCTGTGCG-1,0.0,0.0,0.0
AAACCCAGTTAGAGAT-1,0.0,0.0,0.0


### Шаг 4.3 — KNN imputation (only for K1 single-cell)

Zeros in `log2(TPM+1)` replaces with average of k=5 nearest neighbors 

In [17]:
log_tpm_imputed_gc = sc.knn_impute_log_tpm(log_tpm_gc, knn_k=5)
log_tpm_imputed = log_tpm_imputed_gc.T

zeros_before = (log_tpm_cells == 0).sum().sum()
zeros_after = (log_tpm_imputed == 0).sum().sum()
print(f"zeros before/after impute: {zeros_before} → {zeros_after}")

Loading KNN reference from /workspace/FINAL_VERSION/final_train/splits/sc_k1/X_train.parquet...
✔ KNN reference ready: 1477 cells × 17392 genes


/workspace/FINAL_VERSION/TOTAL_INFERENCE/preprocessor.py:63: RuntimeWarning: Mean of empty slice
  imputed = np.nanmean(neigh_vals, axis=1)


zeros before/after impute: 100460324 → 42766829


### Шаг 4.4 — Stack prediction (K1 cohort)

For each miRNA: CatBoost + TabM + ResNet → Ridge stack

In [18]:
# A: step by step (as above) + predict
pred_manual = sc.predict(log_tpm_imputed, mirnas=sc.mirnas_for_cohort("K1"))
print("manual K1 predictions:", pred_manual.shape)

# B: in one line using raw (recomended)
pred_k1 = sc.predict_single_cell_knn_imputed(raw)
print("one-shot K1 predictions:", pred_k1.shape)
pred_k1.iloc[:5, :5]

Stack prediction for 79 miRNAs...
Loading final_train stack models...
✔ Stack ready: 168 eligible miRNAs
manual K1 predictions: (6761, 79)
✔ Found length for 17392/17392 genes (100.00%)


/workspace/FINAL_VERSION/TOTAL_INFERENCE/preprocessor.py:63: RuntimeWarning: Mean of empty slice
  imputed = np.nanmean(neigh_vals, axis=1)


Stack prediction for 79 miRNAs...
one-shot K1 predictions: (6761, 79)


,hsa-let-7a-5p,hsa-let-7b-5p,hsa-let-7c-5p,hsa-let-7d-5p,hsa-let-7f-5p
barcode,,,,,
AAACCCAGTAAGCAAT-1,10.686427,6.820682,2.775680,2.459605,9.089899
AAACCCAGTCTGTGCG-1,9.047726,4.754303,3.705170,2.598982,6.573882
AAACCCAGTTAGAGAT-1,10.657149,4.801879,3.347979,2.546984,6.676053
AAACCCATCACCTTGC-1,11.437967,7.588331,3.871061,2.716058,9.672027
AAACGAAAGCGATTCT-1,11.602678,7.786710,3.504331,1.967296,10.365216


## 5. Pseudobulk inference (K = 2, 3, 4, 5, 10)

For pseudobulk:
1. In PCA-space (log1p CPM, top HVG) find K nearest neighbors for each cell
2. Sum K cell raw counts → pseudobulk counts
3. TPM → stack **without** KNN impute

In [19]:
K_PB = 2 # example 
pred_pb_k2 = sc.predict_knn_pseudobulk(raw, K=K_PB)
print(f"PB K={K_PB}:", pred_pb_k2.shape)
print("miRNAs:", list(pred_pb_k2.columns[:5]), "...")
pred_pb_k2.iloc[:5, :5]

✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
PB K=2: (6761, 34)
miRNAs: ['hsa-let-7a-3p', 'hsa-let-7e-5p', 'hsa-mir-1307-3p', 'hsa-mir-130b-3p', 'hsa-mir-132-3p'] ...


,hsa-let-7a-3p,hsa-let-7e-5p,hsa-mir-1307-3p,hsa-mir-130b-3p,hsa-mir-132-3p
barcode,,,,,
AAACCCAGTAAGCAAT-1,4.515994,10.074647,11.668998,7.086838,8.292664
AAACCCAGTCTGTGCG-1,4.772171,10.725249,11.403771,6.552921,9.728805
AAACCCAGTTAGAGAT-1,4.431978,9.889214,10.230139,6.731352,7.522930
AAACCCATCACCTTGC-1,4.260909,8.664412,11.400888,6.888928,7.773335
AAACGAAAGCGATTCT-1,4.831328,9.832188,11.548423,6.671389,9.048791


## 8. All in one method — `predict_all`

One method for all **168 eligible** miRNA: K1 → single-cell + KNN, K2…K10 → pseudobulk

In [6]:
SAMPLES = ["RCC_S1", "RCC_S2", "RCC_S3", "RCC_S4", "RCC_S5"]

for sample in SAMPLES:

    INPUT_PATH = RCC_DIR / f"{sample}.csv"
    raw = pd.read_csv(INPUT_PATH)
    raw = raw.set_index("barcode")
    
    pred_all = sc.predict_all(raw)
    out_pb = OUTPUT_DIR / f"{sample}.csv"
    pred_all.to_csv(out_pb)


Full inference: 6761 cells, 168 eligible miRNAs
  K1 single-cell + KNN impute: 79 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Loading KNN reference from /workspace/FINAL_VERSION/final_train/splits/sc_k1/X_train.parquet...
✔ KNN reference ready: 1477 cells × 17392 genes


/workspace/FINAL_VERSION/TOTAL_INFERENCE/preprocessor.py:63: RuntimeWarning: Mean of empty slice
  imputed = np.nanmean(neigh_vals, axis=1)


Stack prediction for 79 miRNAs...
Loading final_train stack models...
✔ Stack ready: 168 eligible miRNAs
  K2 KNN pseudobulk (K=2): 34 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
  K3 KNN pseudobulk (K=3): 30 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
  K4 KNN ps

/workspace/FINAL_VERSION/TOTAL_INFERENCE/preprocessor.py:63: RuntimeWarning: Mean of empty slice
  imputed = np.nanmean(neigh_vals, axis=1)


Stack prediction for 79 miRNAs...
  K2 KNN pseudobulk (K=2): 34 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
  K3 KNN pseudobulk (K=3): 30 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔

...

Stack prediction for 79 miRNAs...
  K2 KNN pseudobulk (K=2): 34 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
  K3 KNN pseudobulk (K=3): 30 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔

/workspace/FINAL_VERSION/TOTAL_INFERENCE/preprocessor.py:63: RuntimeWarning: Mean of empty slice
  imputed = np.nanmean(neigh_vals, axis=1)


Stack prediction for 79 miRNAs...
  K2 KNN pseudobulk (K=2): 34 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
  K3 KNN pseudobulk (K=3): 30 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔

/workspace/FINAL_VERSION/TOTAL_INFERENCE/preprocessor.py:63: RuntimeWarning: Mean of empty slice
  imputed = np.nanmean(neigh_vals, axis=1)


Stack prediction for 79 miRNAs...
  K2 KNN pseudobulk (K=2): 34 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 34 miRNAs...
  K3 KNN pseudobulk (K=3): 30 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 30 miRNAs...
  K4 KNN pseudobulk (K=4): 10 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 10 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 10 miRNAs...
✔ Found length for 17392/17392 genes (100.00%